# Resume-JD Deep Neural Network Match Scorer

**Author:** Shuvam Saren
**Program:** M.Tech, Computer Science & Data Processing
**Department of Mathematics, IIT Kharagpur**

A TensorFlow/Keras **Siamese BiLSTM** that reads a candidate resume and a job
description with a *shared* encoder and predicts a three-way match:

- `0 = Weak Match`
- `1 = Medium Match`
- `2 = Strong Match`

### How the three classes are built (read this)

The source `train.jsonl` gives, per record, a job description, a `Resume-matched`
resume, a weakened `Resume-unmatched` resume, and a ground-truth `Skills` list.
From each record we make three supervised pairs:

| Class | Pair | Nature |
|---|---|---|
| **Strong (2)** | JD + `Resume-matched` | true positive |
| **Medium (1)** | JD + `Resume-unmatched` | **hard** negative: same candidate, a few skills/dates removed |
| **Weak (0)** | JD + a donor resume from another record | **easy** negative, chosen for *low skill overlap* so the label is honest |

Because Medium vs Strong is near-identical text, **accuracy alone is misleading**
(the easy Weak class inflates it). We report **macro-F1** and the confusion
matrix and watch the Medium/Strong rows.

## 0. Imports and reproducibility

In [ ]:
# If running in Google Colab, upload train.jsonl when prompted or place it at
# /content/train.jsonl (the file is the uploaded train.txt renamed to .jsonl).
import os, json, random, re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input, TextVectorization, Embedding, Bidirectional, LSTM,
    GlobalMaxPooling1D, Dense, Dropout, Concatenate, Lambda, Multiply,
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

OUTPUT_DIR = Path("outputs"); MODEL_DIR = Path("models")
OUTPUT_DIR.mkdir(exist_ok=True); MODEL_DIR.mkdir(exist_ok=True)
LABEL_NAMES = {0: "Weak Match", 1: "Medium Match", 2: "Strong Match"}

## 1. Load the JSONL dataset

In [ ]:
DATA_CANDIDATES = [
    Path("/content/train.jsonl"),
    Path("/content/drive/MyDrive/train.jsonl"),
    Path("data/train.jsonl"),
    Path("data/train_sample.jsonl"),   # small bundled sample for a quick run
    Path("train.jsonl"),
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        DATA_PATH = Path(next(iter(uploaded.keys())))
    except Exception as exc:
        raise FileNotFoundError("Upload train.jsonl or set DATA_PATH.") from exc

def read_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

records = read_jsonl(DATA_PATH)
print("Using dataset:", DATA_PATH)
print("Source rows:", len(records))
print("Fields:", list(records[0].keys()))

## 2. Build honest Weak / Medium / Strong pairs

The Weak donor is picked to have **low ground-truth skill overlap** (Jaccard
< 0.10) with the target JD, so a Weak label genuinely means *poor match* rather
than *random pairing*. We also carry each JD's ground-truth skills forward for
the explanation and evaluation steps.

In [ ]:
SOURCE_ROW_LIMIT = None   # None -> all rows. Set e.g. 3000 for a fast Colab run.
WEAK_MAX_SKILL_OVERLAP = 0.10
WEAK_MAX_TRIES = 12

def clean_text(text):
    return re.sub(r"\s+", " ", str(text)).strip()

def skill_set(record):
    return {str(s).strip().lower() for s in (record.get("Skills") or []) if str(s).strip()}

def jaccard(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

def build_labeled_pairs(source_records, row_limit=None, seed=SEED):
    if row_limit is not None:
        source_records = source_records[:row_limit]
    rng = random.Random(seed)
    n = len(source_records)
    jd_texts = [clean_text(r["Job-Description"]) for r in source_records]
    matched = [clean_text(r["Resume-matched"]) for r in source_records]
    unmatched = [clean_text(r["Resume-unmatched"]) for r in source_records]
    skills = [skill_set(r) for r in source_records]

    def weak_donor(i):
        best_j, best_ov = None, 1.0
        for _ in range(WEAK_MAX_TRIES):
            j = rng.randrange(n)
            if j == i:
                continue
            ov = jaccard(skills[i], skills[j])
            if ov < best_ov:
                best_ov, best_j = ov, j
            if ov <= WEAK_MAX_SKILL_OVERLAP:
                return j
        return best_j if best_j is not None else (i + 1) % n

    rows = []
    for i in range(n):
        jd = jd_texts[i]; jd_sk = sorted(skills[i]); w = weak_donor(i)
        rows.append({"resume_text": matched[i],   "job_description": jd, "match_label": 2, "jd_skills": jd_sk})
        rows.append({"resume_text": unmatched[i], "job_description": jd, "match_label": 1, "jd_skills": jd_sk})
        rows.append({"resume_text": matched[w],   "job_description": jd, "match_label": 0, "jd_skills": jd_sk})
    return pd.DataFrame(rows)

df = build_labeled_pairs(records, row_limit=SOURCE_ROW_LIMIT, seed=SEED)
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
print("Total pairs:", df.shape)
print(df["match_label"].map(LABEL_NAMES).value_counts())

### Sanity check: does the skill gradient match the labels?

In [ ]:
def coverage(resume, jd_skills):
    r = str(resume).lower()
    rt = set(re.findall(r"[a-z0-9]+", r))
    if not jd_skills:
        return None
    hit = 0
    for s in jd_skills:
        toks = [t for t in re.findall(r"[a-z0-9]+", s) if len(t) > 3]
        if s in r or (toks and all(any(w.startswith(t[:4]) for w in rt) for t in toks)):
            hit += 1
    return hit / len(jd_skills)

for lbl in [2, 1, 0]:
    sub = df[df.match_label == lbl].head(300)
    covs = [c for c in (coverage(r.resume_text, r.jd_skills) for _, r in sub.iterrows()) if c is not None]
    print(f"{LABEL_NAMES[lbl]:13s} mean JD-skill coverage in resume: {np.mean(covs):.3f}")
# Expect a clear gradient: Strong > Medium > Weak.

## 3. Train / validation / test split (stratified)

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.20, random_state=SEED, stratify=df["match_label"])
val_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["match_label"])
print("Train:", train_df.shape, "| Val:", val_df.shape, "| Test:", test_df.shape)

## 4. Text vectorization (shared vocabulary)

In [ ]:
MAX_TOKENS = 30000
MAX_LEN = 320
EMBED_DIM = 128

vectorizer = TextVectorization(
    max_tokens=MAX_TOKENS, output_mode="int",
    output_sequence_length=MAX_LEN, standardize="lower_and_strip_punctuation",
)
vectorizer.adapt(pd.concat([train_df["resume_text"], train_df["job_description"]]).to_numpy())
vocab_size = len(vectorizer.get_vocabulary())
print("Vocabulary size:", vocab_size)

def make_dataset(dataframe, shuffle=False, batch_size=32):
    resume = dataframe["resume_text"].astype(str).to_numpy()
    jd = dataframe["job_description"].astype(str).to_numpy()
    labels = dataframe["match_label"].astype("int32").to_numpy()
    ds = tf.data.Dataset.from_tensor_slices(((resume, jd), labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=SEED)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

BATCH_SIZE = 32
train_ds = make_dataset(train_df, shuffle=True, batch_size=BATCH_SIZE)
val_ds = make_dataset(val_df, batch_size=BATCH_SIZE)
test_ds = make_dataset(test_df, batch_size=BATCH_SIZE)

## 5. Siamese BiLSTM model

Two towers share one `Embedding` + `BiLSTM` so resume and JD land in the same
space. Comparison block = `[u, v, |u-v|, u*v]`; light L2 on the dense head.

In [ ]:
def build_siamese_bilstm_model(vocab_size, embed_dim=128, lstm_units=64,
                               dropout_rate=0.3, l2_reg=1e-5, lr=1e-3):
    resume_input = Input(shape=(), dtype=tf.string, name="resume_text")
    jd_input = Input(shape=(), dtype=tf.string, name="job_description")

    embedding = Embedding(vocab_size, embed_dim, mask_zero=True, name="shared_embedding")
    bilstm = Bidirectional(LSTM(lstm_units, return_sequences=True), name="shared_bilstm")
    pool = GlobalMaxPooling1D(name="global_max_pool")

    def encode(x):
        x = vectorizer(x); x = embedding(x); x = bilstm(x); return pool(x)

    u, v = encode(resume_input), encode(jd_input)
    abs_diff = Lambda(lambda t: tf.abs(t[0] - t[1]), name="abs_difference")([u, v])
    product = Multiply(name="elementwise_product")([u, v])
    features = Concatenate(name="comparison_features")([u, v, abs_diff, product])

    reg = l2(l2_reg) if l2_reg else None
    x = Dense(128, activation="relu", kernel_regularizer=reg)(features)
    x = Dropout(dropout_rate)(x)
    x = Dense(64, activation="relu", kernel_regularizer=reg)(x)
    out = Dense(3, activation="softmax", name="match_class")(x)

    model = Model([resume_input, jd_input], out)
    model.compile(optimizer=Adam(learning_rate=lr),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

model = build_siamese_bilstm_model(vocab_size, embed_dim=EMBED_DIM)
model.summary()

## 6. Train

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
    ModelCheckpoint(MODEL_DIR / "resume_jd_match_model.keras", monitor="val_loss", save_best_only=True),
    CSVLogger(OUTPUT_DIR / "training_log.csv"),
]
history = model.fit(train_ds, validation_data=val_ds, epochs=12, callbacks=callbacks)

In [ ]:
hist = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(hist["loss"], label="train_loss"); axes[0].plot(hist["val_loss"], label="val_loss")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(hist["accuracy"], label="train_acc"); axes[1].plot(hist["val_accuracy"], label="val_acc")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
plt.tight_layout(); fig.savefig(OUTPUT_DIR / "training_history.png", dpi=160); plt.show()

## 7. Evaluate — macro-F1 first, then the confusion matrix

In [ ]:
y_true = test_df["match_label"].to_numpy()
y_prob = model.predict(
    (tf.constant(test_df["resume_text"].astype(str).tolist(), dtype=tf.string),
     tf.constant(test_df["job_description"].astype(str).tolist(), dtype=tf.string)),
    batch_size=BATCH_SIZE,
)
y_pred = np.argmax(y_prob, axis=1)

macro_f1 = f1_score(y_true, y_pred, average="macro")
print(f"Macro-F1 (primary metric): {macro_f1:.4f}\n")
report = classification_report(y_true, y_pred, target_names=[LABEL_NAMES[i] for i in range(3)], digits=4)
print(report)
(OUTPUT_DIR / "classification_report.txt").write_text(
    f"Macro-F1: {macro_f1:.4f}\n\n" + report, encoding="utf-8")

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=[LABEL_NAMES[i] for i in range(3)],
            yticklabels=[LABEL_NAMES[i] for i in range(3)])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Resume-JD Match Confusion Matrix")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=160); plt.show()
# The Medium<->Strong cell is where the real difficulty shows up.

## 8. Predict and explain one pair (uses ground-truth JD skills)

In [ ]:
def extract_resume_tokens(text):
    return set(re.findall(r"[a-z0-9]+", str(text).lower()))

def skill_covered(skill, resume_lower, resume_tokens):
    if re.search(r"(?<![a-z0-9])" + re.escape(skill) + r"(?![a-z0-9])", resume_lower):
        return True
    toks = [t for t in re.findall(r"[a-z0-9]+", skill) if t not in
            {"and","or","to","of","the","a","in","with","for","ability","skills","skill"}]
    if not toks:
        return False
    def tok_ok(t):
        if len(t) < 4:
            return t in resume_tokens
        return any(w.startswith(t[:4]) or t.startswith(w[:4]) for w in resume_tokens)
    return all(tok_ok(t) for t in toks)

def compare_skills(resume_text, jd_skills):
    rl = str(resume_text).lower(); rt = extract_resume_tokens(resume_text)
    covered = sorted({s for s in jd_skills if skill_covered(s.lower(), rl, rt)})
    missing = sorted({s for s in jd_skills if s.lower() not in {c.lower() for c in covered}})
    return covered, missing

def predict_match(resume_text, job_description, jd_skills=None):
    probs = model.predict(
        (tf.constant([str(resume_text)], dtype=tf.string),
         tf.constant([str(job_description)], dtype=tf.string)), verbose=0)[0]
    cls = int(np.argmax(probs))
    common, missing = compare_skills(resume_text, jd_skills or [])
    return {
        "predicted_label": LABEL_NAMES[cls],
        "confidence": float(probs[cls]),
        "probabilities": {LABEL_NAMES[i]: float(probs[i]) for i in range(3)},
        "common_skills": common, "missing_skills": missing,
    }

sample = test_df.iloc[0]
result = predict_match(sample["resume_text"], sample["job_description"], sample["jd_skills"])
result

In [ ]:
common = ", ".join(result["common_skills"]) or "None found"
missing = ", ".join(result["missing_skills"]) or "None found"
prediction_markdown = f"""# Prediction Result

Predicted class: {result['predicted_label']}
Confidence: {result['confidence']:.2%}

Probabilities:
- Weak Match: {result['probabilities']['Weak Match']:.2%}
- Medium Match: {result['probabilities']['Medium Match']:.2%}
- Strong Match: {result['probabilities']['Strong Match']:.2%}

Common skills: {common}

Missing skills: {missing}

Recommendation: Strengthen resume bullets around the missing requirements and
quantify relevant experience (impact, scale, tools).
"""
print(prediction_markdown)
(OUTPUT_DIR / "prediction_result.md").write_text(prediction_markdown, encoding="utf-8")
(OUTPUT_DIR / "class_mapping.json").write_text(json.dumps(LABEL_NAMES, indent=2), encoding="utf-8")

## 9. Save the final model

In [ ]:
model.save(MODEL_DIR / "resume_jd_match_model_final.keras")
print("Saved:")
print(" -", MODEL_DIR / "resume_jd_match_model.keras", "(best by val_loss)")
print(" -", MODEL_DIR / "resume_jd_match_model_final.keras")
for f in ["training_history.png", "confusion_matrix.png", "classification_report.txt", "prediction_result.md"]:
    print(" -", OUTPUT_DIR / f)